# `ptof_obs_bronze_projection`

## In plain terms
This notebook is step one of the whole monitoring pipeline: it takes a plain, read-only copy of
three raw production data tables and republishes them under standard names/columns in the dev
workspace, so every other notebook can build on the same stable foundation instead of each
re-reading (and possibly re-interpreting) the raw production tables differently. Nothing in
here alerts on anything by itself — think of it as photocopying the source-of-truth so
detectors downstream all read from the same copy.

## What this notebook does
This is the **entry point of the ISH/SAA agent observability pipeline**. It projects three
prod-catalog source tables into durable `oil_obs` VIEWs (`v_llm_bronze`, `v_ish_bronze`,
`v_etl_bronze`), adding derived flags and column aliases so every downstream detector reads a
stable contract instead of raw source schemas.

## Source tables (read-only, prod catalog)
- **`mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs`** → `v_llm_bronze`
  One row per generated output. Columns: `id`, `shift_date`, `shift_type`, `batch_nbr`,
  `output_type`, `scheduler_run`, `content` (JSON), `model_config`, `generated_at`, `ingestion_ts`.
  The view aliases `output_type→capability`, `generated_at→called_at`, `content→response_parsed`
  for downstream compatibility.
- **`mq_gmdf_dp_prd.oil.ptof_ish_audit`** → `v_ish_bronze`
  One row per ISH entity change. Schema identical to dev.
- **`mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit`** → `v_etl_bronze`
  One row per ETL task run. Feeds the new ETL health detector.

## Write target
All views are created in **`mq_gmdf_dev.oil_obs`** (dev workspace, full write access).

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `01_bronze_projections` — runs first, before anything else.
- **Downstream:** every other notebook reads `v_llm_bronze`, `v_ish_bronze`, and/or `v_etl_bronze`
  instead of the source tables directly.

In [ ]:
%sql
-- v_llm_bronze: the single durable, queryable view over prod AI shift outputs. Every detector
-- notebook reads this instead of the raw source table. Column aliases (output_type→capability,
-- generated_at→called_at, content→response_parsed) preserve the downstream contract so detectors
-- work unchanged. Being a VIEW over an append-only base table means backtracking queries in the
-- alert notebook always see the full history.
--
-- Source: mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs (~5,900 rows)
-- 4 output_types: saa-display, situational-awareness, sev2-insights, summary
-- content is 100% valid JSON. All rows are successful outputs (no error_msg/success columns).
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_llm_bronze AS
SELECT
    -- pass-through columns from the prod shift outputs table
    p.id,
    p.shift_date,
    p.shift_type,
    p.batch_nbr,

    -- output_type → capability: downstream detectors and capability_registry join on "capability"
    p.output_type AS capability,

    p.scheduler_run,
    p.model_config,

    -- generated_at → called_at: downstream detectors reference "called_at" for time windows
    p.generated_at AS called_at,

    p.ingestion_ts,

    -- content → response_parsed: downstream detectors reference "response_parsed" for JSON
    -- inspection (schema drift, blank output checks, field explosion in nightly baseline)
    p.content AS response_parsed,

    -- is_blank_output: the output record exists but its content is empty, null, or trivially
    -- hollow. This is the core signal for the blank_output detector in ptof_obs_mal_output.ipynb,
    -- wired as CRITICAL in the alert notebook — a silent empty output is worse than no output
    -- because nothing downstream complains.
    CASE
      WHEN p.content IS NULL
        OR length(trim(p.content)) = 0
        OR p.content IN ('{}', '[]', 'null')
        -- summary-specific: a valid JSON envelope with a null/blank how_we_ran field means the
        -- system prompt's "always present" field is missing — structurally blank
        OR (p.output_type = 'summary'
            AND coalesce(trim(try_parse_json(p.content):how_we_ran::string), '') = '')
      THEN true ELSE false
    END AS is_blank_output

FROM mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs p;

-- Columns removed here 2026-09-21 (dead-code audit -- nothing functional deletes without a
-- doc trail, so the reasoning stays here):
--   write_lag_s (seconds between output generation and ingestion) -- its only intended
--     consumer, write_lag_anomalies, was retired the same day: the value is exactly 0 for
--     every row in prod, so there was nothing left to compute it for. See
--     ptof_obs_liveness_detection.ipynb's "Retired detector" note.
--   response_chars, resp_v (pre-parsed JSON variant of content) -- computed speculatively for
--     future detectors that never materialized; no notebook in this pipeline ever reads either
--     column. Every current JSON-field consumer (nightly baseline, schema drift) parses
--     response_parsed itself via from_json(), not this pre-parsed variant.
-- None of the three had a live reader, confirmed by searching every other notebook in this
-- repo. If a future detector needs any of them, recompute from response_parsed/called_at/
-- ingestion_ts directly rather than reviving a column that sat unread.

In [ ]:
%sql
-- v_ish_bronze: durable, append-only view over the ISH change-audit log from prod. Feeds
-- ptof_obs_behavioral_correlation (handover delivery failures and delivery rate).
-- Source: mq_gmdf_dp_prd.oil.ptof_ish_audit (prod schema identical to dev — confirmed).
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_ish_bronze AS
WITH j AS (
  -- j: parse both JSON blobs (the entity's state after and before the change) up front, once,
  -- so every downstream expression can use :field path syntax instead of re-parsing per column.
  SELECT
      id, action, entity_type, entity_id, user_id, user_email, ts, change_summary,
      try_parse_json(after_json)  AS after_v,
      try_parse_json(before_json) AS before_v
  FROM mq_gmdf_dp_prd.oil.ptof_ish_audit
)
SELECT
    id, action, entity_type, entity_id, user_id, user_email, ts, change_summary,
    before_v, after_v,

    -- shift_date_norm / shift_type_norm / batch_id_norm: normalize inconsistent ISH field names
    -- (shift_date vs shift_date_key, etc.) so behavioral correlation can join ISH activity to LLM
    -- outputs (which use shift_date/shift_type/batch_nbr in v_llm_bronze).
    CAST(coalesce(
        after_v:shift_date::string,
        after_v:shift_date_key::string
    ) AS DATE) AS shift_date_norm,
    coalesce(
        after_v:shift_type::string,
        after_v:shift_label::string
    ) AS shift_type_norm,
    coalesce(
        after_v:batch_id::string,
        after_v:content.batch_id::string
    ) AS batch_id_norm,

    -- is_email_disabled_gate: flags the case where a handover email was deliberately not sent
    -- because the EMAIL_ENABLED feature flag was off. Lets handover_delivery_rate distinguish
    -- "the system chose not to send" from "the system tried and failed to send".
    CASE
      WHEN entity_type = 'HandoverEmail'
       AND after_v:sent::boolean = false
       AND after_v:reason::string LIKE '%EMAIL_ENABLED=false%'
      THEN true ELSE false
    END AS is_email_disabled_gate
FROM j;

In [ ]:
%sql
-- v_etl_bronze: pass-through view over the prod ETL pipeline audit log. Feeds the new
-- etl_pipeline_health detector in ptof_obs_liveness_detection.
-- The upstream ETL refreshes ~19 source tables every 10-15 min; if any task fails, the SAA
-- agent is running on stale data but still producing outputs — capability_silence and
-- pipeline_heartbeat won't fire, making this the only early warning.
-- Source: mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit (~25,000 rows, 1 historical failure)
CREATE OR REPLACE VIEW mq_gmdf_dev.oil_obs.v_etl_bronze AS
SELECT
    run_id,
    run_timestamp,
    table_or_view,
    task_name,
    operation,
    status,
    rows_written,
    total_rows,
    error_message,
    duration_seconds,
    ingestion_ts
FROM mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit;